# OdorNet SEA Processing Pipeline

This notebook reproduces the deterministic OdorNet data-classification workflow from repository-local files. The workflow is organized as SEA: Statistical co-occurrence, Expert correction, and AI-assisted semantic alignment.

The notebook has two goals:

1. Make the taxonomy construction logic auditable from released metadata.
2. Regenerate the full molecule-label table from source-level annotations and compare it against `data/processed/full_dataset.csv` by `SMILES`.


## 0. Setup

The notebook uses only local files and reusable code under `src/odornet/`.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from odornet.datasets import LABEL_COLUMNS, load_odornet, load_source_metadata
from odornet.sea import (
    attach_main_labels,
    build_double_drop_matrix,
    build_reverse_mapping,
    load_main_label_mapping,
)

plt.rcParams.update({"figure.dpi": 130, "axes.grid": True})

RAW_SOURCE_PATH = ROOT / "data" / "raw" / "merged_8892_cleaned_251230.pkl"
AI_MAPPING_PATH = ROOT / "data" / "metadata" / "olfactory_classification_strong_weak.json"
SPECIALIST_MAPPING_PATH = ROOT / "data" / "metadata" / "final_specialist_label_mapping.json"
FULL_PATH = ROOT / "data" / "processed" / "full_dataset.csv"

print(f"Repository root: {ROOT}")


## 1. Load Inputs

The raw pickle contains source-level records. The released full dataset is used only as the reference table for the final comparison. Tables are joined and compared by `SMILES`, not by row order.


In [ ]:
source_df = load_source_metadata(root=ROOT, metadata_path=RAW_SOURCE_PATH)
released_full = load_odornet("full", root=ROOT)
released_train = load_odornet("train", root=ROOT)
released_validation = load_odornet("test", root=ROOT)

summary = pd.DataFrame(
    [
        {"table": "raw_source", "rows": len(source_df), "columns": source_df.shape[1]},
        {"table": "released_full", "rows": len(released_full), "columns": released_full.shape[1]},
        {"table": "released_train", "rows": len(released_train), "columns": released_train.shape[1]},
        {"table": "released_validation", "rows": len(released_validation), "columns": released_validation.shape[1]},
    ]
)
display(summary)
print("SMILES sets match raw vs released full:", set(source_df.SMILES) == set(released_full.SMILES))
print("Train/validation overlap:", len(set(released_train.SMILES) & set(released_validation.SMILES)))


## 2. S: Statistical Co-occurrence

The statistical component summarizes descriptor frequency and conditional co-occurrence probability `P(B|A)` from source-level processed labels. This step is descriptive: it shows which odor descriptors are frequent and which descriptors tend to appear together before they are assigned to primary categories.


In [ ]:
transactions = source_df["Processed_Labels"].tolist()
all_terms = sorted({term for labels in transactions for term in labels})
one_hot = pd.DataFrame(0, index=np.arange(len(transactions)), columns=all_terms, dtype=np.int32)
for row_idx, labels in enumerate(transactions):
    one_hot.loc[row_idx, labels] = 1

co_counts = one_hot.T @ one_hot
term_counts = pd.Series(np.diag(co_counts), index=co_counts.index, name="count")
co_probability = co_counts.div(term_counts.replace(0, np.nan), axis=0).fillna(0.0)
np.fill_diagonal(co_probability.values, 0.0)

top_terms = term_counts.sort_values(ascending=False).head(25)
display(top_terms.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_terms.sort_values().plot(kind="barh", ax=axes[0], color="#4464ad")
axes[0].set_title("Top odor descriptors")
axes[0].set_xlabel("Molecule count")

top20 = top_terms.index[:20]
image = axes[1].imshow(co_probability.loc[top20, top20], cmap="YlGnBu", vmin=0, vmax=1)
axes[1].set_xticks(range(len(top20)))
axes[1].set_xticklabels(top20, rotation=90, fontsize=7)
axes[1].set_yticks(range(len(top20)))
axes[1].set_yticklabels(top20, fontsize=7)
axes[1].set_title("Conditional co-occurrence P(B|A)")
fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
min_count = 30
min_confidence = 0.45
G = nx.DiGraph()
for antecedent in co_probability.index:
    if term_counts[antecedent] < min_count:
        continue
    for consequent in co_probability.columns:
        if antecedent == consequent or term_counts[consequent] < min_count:
            continue
        confidence = co_probability.loc[antecedent, consequent]
        if confidence >= min_confidence:
            G.add_edge(antecedent, consequent, weight=float(confidence))

print(f"Graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")
top_nodes = [node for node, _ in sorted(G.degree, key=lambda item: item[1], reverse=True)[:30]]
H = G.subgraph(top_nodes).copy()

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(H, seed=959, k=0.7)
edge_widths = [1 + 3 * H[u][v]["weight"] for u, v in H.edges]
nx.draw_networkx_nodes(H, pos, node_size=450, node_color="#f0c987")
nx.draw_networkx_edges(H, pos, width=edge_widths, alpha=0.45, arrows=True, arrowsize=10)
nx.draw_networkx_labels(H, pos, font_size=8)
plt.title("Descriptor co-occurrence graph")
plt.axis("off")
plt.show()


## 3. E: Expert Correction

The expert classification was normalized by synonym replacement and terminology harmonization, then saved as `odornet/data/metadata/final_specialist_label_mapping.json`. The expert terms were reviewed against the Fragrantica notes vocabulary at https://www.fragrantica.com/notes/. This file is used as a correction layer rather than as a complete taxonomy.

The comparison below makes two points explicit:

1. The expert mapping overlaps with the statistically observed descriptor vocabulary and supports several major odor families.
2. The expert mapping does not include every primary category and does not classify many secondary descriptors observed in the source data.


In [ ]:
with SPECIALIST_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    specialist_mapping = json.load(handle)
with AI_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    ai_mapping = json.load(handle)

source_vocab = set(all_terms)
expert_vocab = {term for terms in specialist_mapping.values() for term in terms}
ai_strong_vocab = {
    term
    for category, terms in ai_mapping.items()
    if not category.endswith("_weak") and category != "not any type"
    for term in terms
}
ai_weak_vocab = {
    term
    for category, terms in ai_mapping.items()
    if category.endswith("_weak")
    for term in terms
}

expert_missing_categories = [category for category in LABEL_COLUMNS if category not in specialist_mapping]
coverage_summary = pd.DataFrame(
    [
        {"item": "source descriptor vocabulary", "count": len(source_vocab)},
        {"item": "expert descriptors", "count": len(expert_vocab)},
        {"item": "expert descriptors observed in source vocabulary", "count": len(expert_vocab & source_vocab)},
        {"item": "source descriptors without expert assignment", "count": len(source_vocab - expert_vocab)},
        {"item": "expert descriptors also in AI strong mapping", "count": len(expert_vocab & ai_strong_vocab)},
        {"item": "expert descriptors also in AI weak mapping", "count": len(expert_vocab & ai_weak_vocab)},
    ]
)
display(coverage_summary)
print("Expert categories present:", sorted(specialist_mapping))
print("Primary categories absent from expert mapping:", expert_missing_categories)

category_summary = []
for category in LABEL_COLUMNS:
    expert_terms = set(specialist_mapping.get(category, []))
    category_summary.append(
        {
            "primary_category": category,
            "expert_terms": len(expert_terms),
            "expert_terms_seen_in_source": len(expert_terms & source_vocab),
            "ai_strong_terms": len(set(ai_mapping.get(category, []))),
            "ai_weak_terms": len(set(ai_mapping.get(f"{category}_weak", []))),
        }
    )
category_summary = pd.DataFrame(category_summary)
display(category_summary)

plot_df = category_summary.set_index("primary_category")[["expert_terms", "ai_strong_terms", "ai_weak_terms"]]
plot_df.plot(kind="bar", figsize=(12, 4), color=["#d95d39", "#4464ad", "#70a288"])
plt.title("Expert and AI-aligned descriptor coverage by primary category")
plt.ylabel("Descriptor count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 4. Primary Categories Chosen from S and E

Based on the statistical descriptor structure and the normalized expert correction layer, OdorNet uses the following primary categories. These are the final label columns in the released full dataset:

1. `animalic&ambery`
2. `sweety&gourmand`
3. `floral`
4. `fruity&vegetable`
5. `pungent&disagreetable`
6. `green&herbal`
7. `nutty`
8. `woody&mossy`
9. `resinous&balsamic`
10. `cooked`
11. `odorless`
12. `spice`


## 5. A: AI-Assisted Semantic Alignment

Descriptors that were not fully resolved by S and E were aligned to the primary categories with AI-assisted semantic review. The released result is `odornet/data/metadata/olfactory_classification_strong_weak.json`.

In this metadata file, a strong descriptor is a high-consensus assignment to a primary category. A weak descriptor is a lower-consensus but plausible assignment. During full-table generation, weak evidence does not create a positive label by itself; instead, it prevents a confident negative label and therefore produces an unresolved value when source evidence is ambiguous.


In [ ]:
strong_weak_summary = []
for category in LABEL_COLUMNS:
    strong_weak_summary.append(
        {
            "primary_category": category,
            "strong_descriptor_count": len(set(ai_mapping.get(category, []))),
            "weak_descriptor_count": len(set(ai_mapping.get(f"{category}_weak", []))),
        }
    )
strong_weak_summary = pd.DataFrame(strong_weak_summary)
display(strong_weak_summary)
print("Descriptors retained as not any type:", len(ai_mapping.get("not any type", [])))

strong_weak_summary.set_index("primary_category").plot(
    kind="barh",
    figsize=(8, 5),
    color=["#4464ad", "#70a288"],
)
plt.title("AI-assisted strong and weak descriptor assignments")
plt.xlabel("Descriptor count")
plt.tight_layout()
plt.show()


## 6. Generate the Full Label Matrix

This section follows the released processing logic used by `codex_log/reference_inputs/step1_dataset_processing_weak_perfect_test.py`:

1. Merge the AI-assisted strong/weak mapping with the expert correction mapping.
2. Add `odorless` as its own descriptor-to-category mapping.
3. Map each source record's `Processed_Labels` to `Main_label`.
4. Apply the Double-Drop rule across all source records for each molecule.
5. Resolve direct `odorless` conflicts by setting the conflicting molecule-label entries to unresolved values.

The generated table is then compared with the provided `full_dataset.csv` by `SMILES` and by every label column.


In [ ]:
final_mapping = load_main_label_mapping(AI_MAPPING_PATH, SPECIALIST_MAPPING_PATH)
reverse_mapping = build_reverse_mapping(final_mapping)
mapped_source_df = attach_main_labels(source_df, reverse_mapping)
generated_full = build_double_drop_matrix(mapped_source_df, target_labels=LABEL_COLUMNS)

source_column = source_df[["SMILES", "Source"]].copy()
source_column["Source"] = source_column["Source"].map(
    lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True)
)
merged_full = generated_full.merge(source_column, on="SMILES", how="left", validate="one_to_one")
merged_full = merged_full[["SMILES", "Source", *LABEL_COLUMNS]]

print("Generated full shape:", generated_full.shape)
print("Merged full shape:", merged_full.shape)
print("Rows with Source:", merged_full["Source"].notna().sum())
display(merged_full.head(2))


In [ ]:
def compare_by_smiles(generated: pd.DataFrame, released: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = generated[["SMILES", *LABEL_COLUMNS]].merge(
        released[["SMILES", *LABEL_COLUMNS]],
        on="SMILES",
        how="outer",
        suffixes=("_generated", "_released"),
        indicator=True,
    )
    rows = []
    diff_records = []
    both_sides = merged["_merge"].eq("both")
    for label in LABEL_COLUMNS:
        generated_values = pd.to_numeric(merged[f"{label}_generated"], errors="coerce")
        released_values = pd.to_numeric(merged[f"{label}_released"], errors="coerce")
        equal_values = (generated_values == released_values) | (
            generated_values.isna() & released_values.isna()
        )
        diff_mask = ~(both_sides & equal_values)
        rows.append(
            {
                "label": label,
                "different_rows": int(diff_mask.sum()),
                "generated_positive": int((generated_values == 1.0).sum()),
                "released_positive": int((released_values == 1.0).sum()),
                "generated_nan": int(generated_values.isna().sum()),
                "released_nan": int(released_values.isna().sum()),
            }
        )
        for _, row in merged.loc[diff_mask, ["SMILES", f"{label}_generated", f"{label}_released", "_merge"]].iterrows():
            diff_records.append(
                {
                    "SMILES": row["SMILES"],
                    "label": label,
                    "generated": row[f"{label}_generated"],
                    "released": row[f"{label}_released"],
                    "merge_status": row["_merge"],
                }
            )
    return pd.DataFrame(rows), pd.DataFrame(diff_records)


comparison, diff_records = compare_by_smiles(merged_full, released_full)
display(comparison)
print("SMILES only in generated:", int((~merged_full["SMILES"].isin(released_full["SMILES"])).sum()))
print("SMILES only in released:", int((~released_full["SMILES"].isin(merged_full["SMILES"])).sum()))
print("Total molecule-label differences:", int(comparison["different_rows"].sum()))
print("Unique molecules with any label difference:", diff_records["SMILES"].nunique() if not diff_records.empty else 0)

assert int((~merged_full["SMILES"].isin(released_full["SMILES"])).sum()) == 0
assert int((~released_full["SMILES"].isin(merged_full["SMILES"])).sum()) == 0
assert int(comparison["different_rows"].sum()) == 0


## 7. Released Split Sanity Check

The train and validation files are fixed release artifacts. The detailed split strategy is documented in `notebooks/odornet_split_strategy.ipynb`; this section only checks that the released split covers the full molecule set without overlap.


In [ ]:
print("Train/validation overlap:", len(set(released_train.SMILES) & set(released_validation.SMILES)))
print(
    "Train+validation union equals released full:",
    set(released_train.SMILES) | set(released_validation.SMILES) == set(released_full.SMILES),
)

split_counts = pd.DataFrame(
    {
        "full": released_full[LABEL_COLUMNS].apply(pd.to_numeric, errors="coerce").sum(),
        "train": released_train[LABEL_COLUMNS].apply(pd.to_numeric, errors="coerce").sum(),
        "validation": released_validation[LABEL_COLUMNS].apply(pd.to_numeric, errors="coerce").sum(),
    }
).sort_values("full", ascending=False)
display(split_counts)

split_counts.plot(kind="bar", figsize=(12, 4), color=["#333333", "#52796f", "#f4a259"])
plt.title("Positive labels by split")
plt.ylabel("Positive count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
